# Déploiement (MVP) & Perspectives LLM

## Objectif
Ce notebook présente une projection réaliste du modèle champion vers un usage de type MVP (Minimum Viable Product).

## Objectifs
- illustrer un pipeline simple d’inférence,
- définir une sortie exploitable pour un utilisateur ou un reviewer,
- expliciter les garde-fous cliniques,
- discuter le rôle potentiel des LLMs comme extension future.

## Positionnement
Le système reste un outil de **triage textuel non diagnostique**.  
Aucune sortie ne doit être interprétée comme un diagnostic médical.

In [7]:
import sys
from pathlib import Path

current = Path.cwd().resolve()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent
elif (current / "src").exists():
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = current

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config.paths import *

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        PROJECT_ROOT 
    )
)

SRC_DIR = PROJECT_ROOT / "src"

# Mantemos o nome para não partir o código
MODEL_DIR = PROJECT_ROOT / "models"

DATA_DIR = PROJECT_ROOT / "data"
DATA_CLEAN_DIR = DATA_DIR / "clean"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

# Mantemos o nome
CLASSICAL_REPORTS_DIR = TABLES_DIR / "classical"

DEPLOYMENT_TABLES_DIR = TABLES_DIR / "deployment"
DEPLOYMENT_FIGURES_DIR = FIGURES_DIR / "deployment"

FINAL_MODEL_PATH = MODEL_DIR / "best_classical_model.joblib"
CLEAN_DATA_PATH = DATA_CLEAN_DIR / "mental_health_detection_clean.csv"
FINAL_TEST_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "final_test_metrics.csv"

for directory in [
    MODEL_DIR,
    DATA_CLEAN_DIR,
    CLASSICAL_REPORTS_DIR,
    DEPLOYMENT_TABLES_DIR,
    DEPLOYMENT_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Paths ready")
print("MODEL_DIR:", MODEL_DIR)
print("DEPLOYMENT_TABLES_DIR:", DEPLOYMENT_TABLES_DIR)
print("DEPLOYMENT_FIGURES_DIR:", DEPLOYMENT_FIGURES_DIR)

✅ Paths ready
MODEL_DIR: C:\Users\anafi\Desktop\final_project_jedha\models
DEPLOYMENT_TABLES_DIR: C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment
DEPLOYMENT_FIGURES_DIR: C:\Users\anafi\Desktop\final_project_jedha\reports\figures\deployment


In [9]:
# ============================================================
# PHASE 0 — IMPORTS & GLOBAL SETTINGS
# ============================================================

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TEXT_COL = "body"
MASKED_COL = "body_masked"
TARGET_COL = "category"

PLOTLY_TEMPLATE = "plotly_white"

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 100)

print("✅ Imports loaded")
print(f"TEXT_COL   = {TEXT_COL}")
print(f"MASKED_COL = {MASKED_COL}")
print(f"TARGET_COL = {TARGET_COL}")

✅ Imports loaded
TEXT_COL   = body
MASKED_COL = body_masked
TARGET_COL = category


## Chargement du modèle final et des données

Cette section charge le modèle champion exporté depuis le benchmark classique, ainsi que le dataset propre utilisé dans le projet.

In [10]:
# ============================================================
# PHASE 1 — LOAD MODEL AND DATA
# ============================================================

final_model = joblib.load(FINAL_MODEL_PATH)
df_clean = pd.read_csv(CLEAN_DATA_PATH)
final_test_metrics = pd.read_csv(FINAL_TEST_RESULTS_PATH)

required_cols = [TEXT_COL, MASKED_COL, TARGET_COL]
missing_cols = [col for col in required_cols if col not in df_clean.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in df_clean: {missing_cols}")

print("✅ Final model loaded")
print("✅ Clean dataset loaded")
print("Model type:", type(final_model))
print("df_clean shape:", df_clean.shape)

display(final_test_metrics)

c:\Users\anafi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\anafi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\anafi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle esti

✅ Final model loaded
✅ Clean dataset loaded
Model type: <class 'sklearn.pipeline.Pipeline'>
df_clean shape: (11271, 3)


,champion_model,text_variant,f1_macro,recall_macro,critical_recall
0,LinearSVC_balanced,raw,0.778927,0.778686,0.698068


In [11]:
# ============================================================
# PHASE 2 — REBUILD CHAMPION CONTEXT
# ============================================================

if {"metric", "value"}.issubset(final_test_metrics.columns):
    metric_lookup = dict(zip(final_test_metrics["metric"], final_test_metrics["value"]))
    CHAMPION_MODEL_NAME = metric_lookup.get("champion_model", "best_classical_model")
    CHAMPION_TEXT_VARIANT = metric_lookup.get("text_variant", "raw")
else:
    first_row = final_test_metrics.iloc[0].to_dict()
    CHAMPION_MODEL_NAME = first_row.get("champion_model", "best_classical_model")
    CHAMPION_TEXT_VARIANT = first_row.get("text_variant", "raw")

print("✅ Champion context rebuilt")
print("CHAMPION_MODEL_NAME :", CHAMPION_MODEL_NAME)
print("CHAMPION_TEXT_VARIANT:", CHAMPION_TEXT_VARIANT)

✅ Champion context rebuilt
CHAMPION_MODEL_NAME : LinearSVC_balanced
CHAMPION_TEXT_VARIANT: raw


## Démonstration d’inférence

Cette section illustre le comportement du modèle champion sur quelques exemples du corpus, dans une logique proche d’un futur MVP.

In [12]:
def prepare_text_for_inference(text):
    """
    Basic safety preparation for single-text inference.
    """
    if pd.isna(text):
        return ""
    return str(text).strip()


def predict_single_text(model, text):
    """
    Run single-text inference with the trained final model.
    """
    prepared_text = prepare_text_for_inference(text)
    prediction = model.predict([prepared_text])[0]

    return {
        "input_text": prepared_text,
        "predicted_label": prediction,
    }

In [13]:
# ============================================================
# PHASE 3 — SAMPLE INFERENCE DEMO
# ============================================================

sample_df = df_clean.sample(5, random_state=RANDOM_STATE).reset_index(drop=True)

demo_rows = []
for _, row in sample_df.iterrows():
    source_text = row[TEXT_COL] if CHAMPION_TEXT_VARIANT == "raw" else row[MASKED_COL]

    pred = predict_single_text(
        final_model,
        source_text,
    )

    demo_rows.append({
        "true_label": row[TARGET_COL],
        "predicted_label": pred["predicted_label"],
        "text_used_for_model": source_text,
    })

deployment_demo_df = pd.DataFrame(demo_rows)

display(deployment_demo_df.head(10))

deployment_demo_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "sample_inference_demo.csv",
    index=False,
)

print("✅ Sample inference demo saved")

,true_label,predicted_label,text_used_for_model
0,Anxiety,Anxiety,So I was excited because within the last few months I made a friend and weve become close Weve hung out and texted a bunch We bonded over our love of fanfiction and fandom in general This is a big deal for me because I have like 1 other friend\n\nWe were texting about Harry Potter tattoos She ha...
1,ADHD,ADHD,does anyone know if as you get older you can swap adderall for coffee i know that the effects of focus are similar but very short term in comparison to adderall xr i am just trying to find ways to start to stop depending on prescriptions to be able to control my train of thought
2,Autism,Autism,Hey all\n\nIm 23 AFAB not sure if I am autistic but I was diagnosed with SPD in my teenage years Lately things have been getting more and more difficult In the last year or so it has become harder and harder for me to cope with physical discomfort among other things Well because of this my last ...
3,Anxiety,Anxiety,I apologise in advance if this sounds like a ventrant\nBut I always have a feeling that Im completely talentless My anxiety has really really stopped me from putting myself out there and being apart of stuff such as internships community service etc etc whoever I talk to has a story about themse...
4,Depression,Depression,I reached a state that my thoughts just get placed in quarantine Its become such existential nightmare that I feel I have even removed my consciousness of that area of my mind Im losing my humanity as the safe areas shrink If I would tell someone my true thoughts I would risk spreading the virus...


✅ Sample inference demo saved


## Architecture fonctionnelle du MVP

Le MVP visé n’est pas un produit médical, mais une interface simple permettant :

- la soumission d’un texte,
- l’exécution du pipeline de prédiction,
- l’affichage d’un label prédit,
- l’ajout d’un avertissement clinique,
- la possibilité d’une revue humaine.

In [14]:
# ============================================================
# PHASE 4 — MVP FUNCTIONAL DESIGN
# ============================================================

mvp_design_df = pd.DataFrame({
    "component": [
        "User input",
        "Text preprocessing",
        "Model inference",
        "Prediction output",
        "Clinical disclaimer",
        "Human review layer",
        "Logging / audit trail",
    ],
    "role": [
        "Collect free-text input",
        "Prepare text in the expected format",
        "Run the champion model on the text",
        "Return the predicted class",
        "Remind non-diagnostic scope",
        "Allow a reviewer to inspect difficult cases",
        "Store predictions and review outcomes",
    ],
})

display(mvp_design_df)

mvp_design_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "mvp_functional_design.csv",
    index=False,
)

print("✅ MVP design table saved")

,component,role
0,User input,Collect free-text input
1,Text preprocessing,Prepare text in the expected format
2,Model inference,Run the champion model on the text
3,Prediction output,Return the predicted class
4,Clinical disclaimer,Remind non-diagnostic scope
5,Human review layer,Allow a reviewer to inspect difficult cases
6,Logging / audit trail,Store predictions and review outcomes


✅ MVP design table saved


## Format de sortie du MVP

Dans une version simple du MVP, la sortie affichée à l’utilisateur ou au reviewer doit rester sobre, compréhensible et prudente.

Elle peut inclure :
- la classe prédite,
- un rappel explicite du cadre non diagnostique,
- une indication de revue humaine recommandée.

In [15]:
# ============================================================
# PHASE 5 — MVP OUTPUT SCHEMA
# ============================================================

mvp_output_schema_df = pd.DataFrame({
    "output_field": [
        "predicted_label",
        "display_message",
        "clinical_warning",
        "human_review_recommended",
    ],
    "description": [
        "Predicted class returned by the champion model",
        "Simple user-facing message summarizing the result",
        "Explicit non-diagnostic warning",
        "Boolean flag indicating whether human review is recommended",
    ],
})

display(mvp_output_schema_df)

mvp_output_schema_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "mvp_output_schema.csv",
    index=False,
)

print("✅ MVP output schema saved")

,output_field,description
0,predicted_label,Predicted class returned by the champion model
1,display_message,Simple user-facing message summarizing the result
2,clinical_warning,Explicit non-diagnostic warning
3,human_review_recommended,Boolean flag indicating whether human review is recommended


✅ MVP output schema saved


In [16]:
# ============================================================
# PHASE 6 — STRUCTURED MVP OUTPUT DEMO
# ============================================================

def build_mvp_output(predicted_label):
    """
    Build a lightweight structured output for MVP display.
    """
    return {
        "predicted_label": predicted_label,
        "display_message": f"Predicted class: {predicted_label}",
        "clinical_warning": (
            "This output is a non-diagnostic triage signal and must not replace clinical judgment."
        ),
        "human_review_recommended": True,
    }

mvp_demo_outputs = []

for _, row in deployment_demo_df.head(5).iterrows():
    output = build_mvp_output(row["predicted_label"])
    output["true_label"] = row["true_label"]
    output["text_used_for_model"] = row["text_used_for_model"]
    mvp_demo_outputs.append(output)

mvp_demo_outputs_df = pd.DataFrame(mvp_demo_outputs)

display(mvp_demo_outputs_df)

mvp_demo_outputs_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "mvp_demo_outputs.csv",
    index=False,
)

print("✅ MVP demo outputs saved")

,predicted_label,display_message,clinical_warning,human_review_recommended,true_label,text_used_for_model
0,Anxiety,Predicted class: Anxiety,This output is a non-diagnostic triage signal and must not replace clinical judgment.,True,Anxiety,So I was excited because within the last few months I made a friend and weve become close Weve hung out and texted a bunch We bonded over our love of fanfiction and fandom in general This is a big deal for me because I have like 1 other friend\n\nWe were texting about Harry Potter tattoos She ha...
1,ADHD,Predicted class: ADHD,This output is a non-diagnostic triage signal and must not replace clinical judgment.,True,ADHD,does anyone know if as you get older you can swap adderall for coffee i know that the effects of focus are similar but very short term in comparison to adderall xr i am just trying to find ways to start to stop depending on prescriptions to be able to control my train of thought
2,Autism,Predicted class: Autism,This output is a non-diagnostic triage signal and must not replace clinical judgment.,True,Autism,Hey all\n\nIm 23 AFAB not sure if I am autistic but I was diagnosed with SPD in my teenage years Lately things have been getting more and more difficult In the last year or so it has become harder and harder for me to cope with physical discomfort among other things Well because of this my last ...
3,Anxiety,Predicted class: Anxiety,This output is a non-diagnostic triage signal and must not replace clinical judgment.,True,Anxiety,I apologise in advance if this sounds like a ventrant\nBut I always have a feeling that Im completely talentless My anxiety has really really stopped me from putting myself out there and being apart of stuff such as internships community service etc etc whoever I talk to has a story about themse...
4,Depression,Predicted class: Depression,This output is a non-diagnostic triage signal and must not replace clinical judgment.,True,Depression,I reached a state that my thoughts just get placed in quarantine Its become such existential nightmare that I feel I have even removed my consciousness of that area of my mind Im losing my humanity as the safe areas shrink If I would tell someone my true thoughts I would risk spreading the virus...


✅ MVP demo outputs saved


## Garde-fous cliniques

Tout MVP dérivé de ce projet doit intégrer des garde-fous explicites pour éviter une mauvaise interprétation des sorties du modèle.

In [17]:
# ============================================================
# PHASE 7 — CLINICAL GUARDRAILS
# ============================================================

clinical_guardrails_df = pd.DataFrame({
    "guardrail": [
        "Non-diagnostic disclaimer",
        "Human review required",
        "No autonomous decision-making",
        "Error monitoring on critical classes",
        "Logging for audit",
        "Restricted intended use",
    ],
    "purpose": [
        "Prevent users from interpreting the output as a diagnosis",
        "Ensure sensitive cases can be reviewed manually",
        "Avoid unsafe automated escalation or dismissal",
        "Track errors involving Bipolar and schizophrenia",
        "Support traceability and future quality review",
        "Keep the system within a triage/research scope",
    ],
})

display(clinical_guardrails_df)

clinical_guardrails_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "clinical_guardrails.csv",
    index=False,
)

print("✅ Clinical guardrails table saved")

,guardrail,purpose
0,Non-diagnostic disclaimer,Prevent users from interpreting the output as a diagnosis
1,Human review required,Ensure sensitive cases can be reviewed manually
2,No autonomous decision-making,Avoid unsafe automated escalation or dismissal
3,Error monitoring on critical classes,Track errors involving Bipolar and schizophrenia
4,Logging for audit,Support traceability and future quality review
5,Restricted intended use,Keep the system within a triage/research scope


✅ Clinical guardrails table saved


In [18]:
clinical_guardrails_plot_df = clinical_guardrails_df.copy()
clinical_guardrails_plot_df["included"] = 1


fig = px.bar(
    clinical_guardrails_plot_df,
    x="guardrail",
    y="included",
    template=PLOTLY_TEMPLATE,
    title="Garde-fous cliniques à intégrer dans le MVP",
)

fig.update_layout(
    xaxis_title="Guardrail",
    yaxis_title="Included",
    title_x=0.5,
    showlegend=False,
)

fig.write_html(str(DEPLOYMENT_FIGURES_DIR / "clinical_guardrails.html"))
fig.show()

print("✅ Clinical guardrails figure saved")

✅ Clinical guardrails figure saved


## Perspectives LLM

Les LLMs ne doivent pas remplacer le classifieur principal, mais peuvent constituer une extension future utile dans certaines fonctions encadrées.

In [19]:
# ============================================================
# PHASE 8 — LLM OUTLOOK
# ============================================================

llm_outlook_df = pd.DataFrame({
    "possible_role": [
        "Case summarization",
        "Reviewer support",
        "Uncertainty explanation",
        "Structured note generation",
        "Human-readable rationale layer",
    ],
    "potential_value": [
        "Summarize long free-text cases for a reviewer",
        "Help organize difficult cases for manual inspection",
        "Explain that outputs remain uncertain and non-diagnostic",
        "Produce standardized review notes",
        "Improve interpretability for downstream review",
    ],
    "main_risk": [
        "Over-summarization of clinically relevant details",
        "Hallucinated interpretations",
        "False sense of reliability",
        "Unsafe or misleading wording",
        "Unsupported explanations not grounded in the classifier",
    ],
})

display(llm_outlook_df)

llm_outlook_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "llm_outlook_table.csv",
    index=False,
)

print("✅ LLM outlook table saved")

,possible_role,potential_value,main_risk
0,Case summarization,Summarize long free-text cases for a reviewer,Over-summarization of clinically relevant details
1,Reviewer support,Help organize difficult cases for manual inspection,Hallucinated interpretations
2,Uncertainty explanation,Explain that outputs remain uncertain and non-diagnostic,False sense of reliability
3,Structured note generation,Produce standardized review notes,Unsafe or misleading wording
4,Human-readable rationale layer,Improve interpretability for downstream review,Unsupported explanations not grounded in the classifier


✅ LLM outlook table saved


## Position retenue sur les LLMs

Dans l’état actuel du projet, l’usage le plus raisonnable des LLMs serait :

- **en aval** du classifieur principal,
- **sous supervision humaine**,
- dans un rôle d’assistance documentaire ou explicative,
- jamais comme moteur autonome de décision clinique.

In [20]:
# ============================================================
# PHASE 9 — FINAL DEPLOYMENT SUMMARY
# ============================================================

deployment_summary_df = pd.DataFrame({
    "item": [
        "Champion model",
        "Champion text variant",
        "Deployment artifact",
        "Primary intended use",
        "Human review required",
        "LLM role recommended",
    ],
    "value": [
        CHAMPION_MODEL_NAME,
        CHAMPION_TEXT_VARIANT,
        str(FINAL_MODEL_PATH),
        "Non-diagnostic text triage support",
        "Yes",
        "Downstream support only",
    ],
})

display(deployment_summary_df)

deployment_summary_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "deployment_final_summary.csv",
    index=False,
)

print("✅ Deployment final summary saved")

,item,value
0,Champion model,LinearSVC_balanced
1,Champion text variant,raw
2,Deployment artifact,C:\Users\anafi\Desktop\final_project_jedha\models\best_classical_model.joblib
3,Primary intended use,Non-diagnostic text triage support
4,Human review required,Yes
5,LLM role recommended,Downstream support only


✅ Deployment final summary saved


## Conclusion

Le modèle champion peut être projeté vers un MVP simple, à condition de respecter plusieurs principes :

- conserver un cadre strictement non diagnostique,
- imposer une supervision humaine,
- documenter clairement les limites,
- éviter toute autonomie clinique,
- réserver les LLMs à un rôle d’assistance encadrée.

Ce notebook clôt donc la dimension produit du projet en reliant :
- le modèle final,
- une logique d’inférence réaliste,
- des garde-fous cliniques,
- une perspective d’évolution future.

In [21]:
# ============================================================
# PHASE 10 — DEPLOYMENT EXPORTS MANIFEST
# ============================================================

deployment_exports_df = pd.DataFrame({
    "artifact": [
        "sample_inference_demo.csv",
        "mvp_functional_design.csv",
        "mvp_output_schema.csv",
        "mvp_demo_outputs.csv",
        "clinical_guardrails.csv",
        "llm_outlook_table.csv",
        "deployment_final_summary.csv",
    ],
    "path": [
        str(DEPLOYMENT_TABLES_DIR / "sample_inference_demo.csv"),
        str(DEPLOYMENT_TABLES_DIR / "mvp_functional_design.csv"),
        str(DEPLOYMENT_TABLES_DIR / "mvp_output_schema.csv"),
        str(DEPLOYMENT_TABLES_DIR / "mvp_demo_outputs.csv"),
        str(DEPLOYMENT_TABLES_DIR / "clinical_guardrails.csv"),
        str(DEPLOYMENT_TABLES_DIR / "llm_outlook_table.csv"),
        str(DEPLOYMENT_TABLES_DIR / "deployment_final_summary.csv"),
    ],
})

display(deployment_exports_df)

deployment_exports_df.to_csv(
    DEPLOYMENT_TABLES_DIR / "deployment_exports_manifest.csv",
    index=False,
)

print("✅ Deployment exports manifest saved")

,artifact,path
0,sample_inference_demo.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\sample_inference_demo.csv
1,mvp_functional_design.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\mvp_functional_design.csv
2,mvp_output_schema.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\mvp_output_schema.csv
3,mvp_demo_outputs.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\mvp_demo_outputs.csv
4,clinical_guardrails.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\clinical_guardrails.csv
5,llm_outlook_table.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\llm_outlook_table.csv
6,deployment_final_summary.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\deployment\deployment_final_summary.csv


✅ Deployment exports manifest saved
